# 2025 DL Lab6: Text Summarization with Seq2Seq Model

Before we start, please put **your name** and **SID** in following format: <br>
Hi I'm 陸仁賈, 314831000.

**Your Answer:**    
Hi I'm 吳禎哲, 313833003.

## Overview
This assignment involves implementing a hybrid sequence-to-sequence model to perform text summarization on the SAMSum and Reddit TIFU datasets.

The model architecture is composed of two main parts:
A pre-trained model utilized as the encoder.
A new decoder which must be implemented from scratch.

The objective is to fine-tune the existing encoder while training the custom decoder from the beginning, enabling the complete model to generate accurate and concise summaries. Performance is measured using the standard summarization metric: ROUGE-L Score.

## Kaggle Competition
Kaggle is an online community of data scientists and machine learning practitioners. Kaggle allows users to find and publish datasets, explore and build models in a web-based data-science environment, work with other data scientists and machine learning engineers, and enter competitions to solve data science challenges.

This assignment use kaggle to calculate your grade.  
Please use this [**LINK**](https://www.kaggle.com/t/efb569a4c0774de681e9f8426cfac364) to join the competition.

## Unzip Data

Unzip dataset.zip

### SAMSum
+ `train` : 14700
+ `val` : 818
+ `test` : 819

### Redit_TIFU
+ `train` : 29498
+ `val` : 4212
+ `test` : 8429

In [1]:
import importlib, sys, subprocess

def _ensure_pkg(mod_name: str, pip_name: str):
    try:
        return importlib.import_module(mod_name)
    except ImportError:
        print(f'[INFO] {mod_name} not found; installing {pip_name}...')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--no-cache-dir', pip_name])
        return importlib.import_module(mod_name)

wandb = _ensure_pkg('wandb', 'wandb')
print('[INFO] wandb available:', wandb.__version__)

try:
    rouge_score = importlib.import_module('rouge_score')
    print('[INFO] rouge-score available')
except ImportError:
    try:
        _ensure_pkg('rouge_score', 'rouge-score')
        print('[INFO] rouge-score installed')
    except Exception as e:
        print('[WARN] Failed to install rouge-score:', e)


[INFO] wandb available: 0.23.0
[INFO] rouge-score available


In [2]:
import csv
import math
import random
from pathlib import Path
from typing import Optional, Tuple, Union, List, Dict
from data_utils import *
import torch
import torch.nn.functional as F
from torch.nn.utils import clip_grad_norm_
from torch.utils.data import ConcatDataset, DataLoader, Dataset, WeightedRandomSampler
from transformers import get_linear_schedule_with_warmup
from tqdm.auto import tqdm
from transformers.tokenization_utils_base import PreTrainedTokenizerBase
from transformer.Const import *
from transformer.Models import Seq2SeqModelWithFlashAttn
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"

MODE = "train"  # set to "predict" for inference
CHECKPOINT_PATH = Path("checkpoints/latest.pt")
BEST_CHECKPOINT_PATH = Path("checkpoints/best.pt")
PREDICT_CHECKPOINT = Path("checkpoints/best.pt")
TIFU_TEST_PATH = Path("dataset/tifu/tifu_test.jsonl")
SAMSUN_TEST_PATH = Path("dataset/samsun/test.csv")
PREDICTION_OUTPUT = Path("result.csv")
MAX_GENERATION_LEN = MAX_TARGET_LEN
TRAIN_EPOCHS = 200
TRAIN_BATCH_SIZE = 128
GLOBAL_SEED = 42
NUM_WORKERS = 4
def set_seed(seed: int) -> None:
    random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


In [3]:
def set_seed(seed: int) -> None:
    random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


## CREATE DATASET
use ConCate dataset to handle multiple datasets situation

In [4]:
def build_dataset(
    path: List[Optional[str]],
    tokenizer: PreTrainedTokenizerBase,
    require_target: bool = True,
) -> Tuple[Optional[Dataset], Optional[List[int]]]:
    if all(p is None for p in path):
        return None, None
    datasets = []
    for p in path:
        if p is not None:
            dataset = SquadSeq2SeqDataset(
                Path(p), tokenizer, max_source_len=MAX_SOURCE_LEN, max_target_len=MAX_TARGET_LEN, require_target=require_target
            )
            datasets.append(dataset)
    total = sum(len(ds) for ds in datasets)
    print(f"Built dataset with {total} samples.")
    sizes = [len(ds) for ds in datasets]
    if len(datasets) == 1:
        return datasets[0], sizes
    return ConcatDataset(datasets), sizes


def build_dataloader(
    source: Union[Optional[Dataset], Optional[str]],
    batch_size: int = 4,
    shuffle: bool = False,
    num_workers: int = 8,
    sample_weights: Optional[List[float]] = None,
) -> Optional[DataLoader]:
    dataset = source
    collator = QACollator  # Don't forget to define QACollator in data_utils.py
    sampler = None
    if sample_weights is not None:
        sampler = WeightedRandomSampler(sample_weights, num_samples=len(sample_weights), replacement=True)
        shuffle = False
    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle if sampler is None else False,
        sampler=sampler,
        collate_fn=collator,
        num_workers=num_workers,
        pin_memory=True,
        persistent_workers=num_workers > 0,
    )


## Main loop of your model

In [5]:
def run_epoch(
    dataloader: DataLoader,
    model: Seq2SeqModelWithFlashAttn,
    device: torch.device,
    optimizer: Optional[torch.optim.Optimizer],
    scheduler: Optional[object],
    pad_id: int,
    max_grad_norm: float,
    train: bool,
    scaler: Optional[torch.amp.GradScaler] = None,
    label_smoothing: float = 0.0,
    use_amp: bool = True,
    amp_dtype: torch.dtype = torch.float16,
    dae_ratio: float = 0.15,
    dae_loss_weight: float = 0.5,
) -> float:
    from contextlib import nullcontext
    model.train(train)
    total_loss = 0.0
    steps = 0
    iterator = tqdm(dataloader, desc="train" if train else "eval", leave=False)
    amp_enabled = use_amp and torch.cuda.is_available()
    amp_context = (
        torch.amp.autocast(device_type='cuda', dtype=amp_dtype) if amp_enabled else nullcontext()
    )

    tokenizer = getattr(model, 'tokenizer', None)
    mask_id = getattr(tokenizer, 'mask_token_id', None)

    for batch in iterator:
        src = batch["src"].to(device)              # [B, S]
        tgt = batch["tgt"].to(device)              # [B, T]
        src_seq_len = batch["src_len"].to(device=device, dtype=torch.int32)
        tgt_seq_len = batch["tgt_len"].to(device=device, dtype=torch.int32)
        if torch.any(tgt_seq_len < 2):
            raise ValueError("Each target sequence must contain at least BOS and EOS tokens.")

        # Prepare decoder inputs/labels on padded tensors
        decoder_input_padded = tgt[:, :-1]
        labels_padded = tgt[:, 1:]
        decoder_seq_len = (tgt_seq_len - 1).clamp_min(0)

        # Pack labels to align with model's packed logits output
        labels_list = []
        for i in range(labels_padded.size(0)):
            L = int(decoder_seq_len[i].item())
            if L > 0:
                labels_list.append(labels_padded[i, :L])
        labels = torch.cat(labels_list, dim=0) if labels_list else labels_padded.new_zeros((0,), dtype=labels_padded.dtype)

        with amp_context:
            logits = model(
                src_input_ids=src,
                trg_input_ids=decoder_input_padded,
                src_seq_len=src_seq_len,
                trg_seq_len=decoder_seq_len,
            )
            loss = F.cross_entropy(
                logits, labels, ignore_index=pad_id, label_smoothing=label_smoothing
            )

            # Auxiliary DAE loss (noising decoder inputs)
            if train and dae_loss_weight > 0.0 and decoder_input_padded.numel() > 0:
                noise = decoder_input_padded.clone()
                B, Tm = noise.shape
                if Tm > 0:
                    # build mask for valid (non-pad, non-special) positions
                    valid = (noise != pad_id)
                    if tokenizer is not None:
                        bos = getattr(tokenizer, 'bos_token_id', getattr(tokenizer, 'cls_token_id', None))
                        eos = getattr(tokenizer, 'eos_token_id', getattr(tokenizer, 'sep_token_id', None))
                        if bos is not None:
                            valid &= (noise != bos)
                        if eos is not None:
                            valid &= (noise != eos)
                    # sample corruption
                    prob = torch.rand_like(noise, dtype=torch.float, device=noise.device)
                    corr = (prob < dae_ratio) & valid
                    if corr.any():
                        if mask_id is not None:
                            noise[corr] = mask_id
                        else:
                            # random token replacement (avoid special range heuristically)
                            vocab = model.output_projection.weight.size(0)
                            rand_tokens = torch.randint(low=0, high=vocab, size=(corr.sum().item(),), device=noise.device, dtype=noise.dtype)
                            noise[corr] = rand_tokens
                    logits_dae = model(
                        src_input_ids=src,
                        trg_input_ids=noise,
                        src_seq_len=src_seq_len,
                        trg_seq_len=decoder_seq_len,
                    )
                    loss_dae = F.cross_entropy(
                        logits_dae, labels, ignore_index=pad_id, label_smoothing=label_smoothing
                    )
                    loss = loss + dae_loss_weight * loss_dae

        if train:
            if scaler is not None and amp_dtype == torch.float16:
                optimizer.zero_grad(set_to_none=True)
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                clip_grad_norm_(model.parameters(), max_grad_norm)
                scaler.step(optimizer)
                scaler.update()
            else:
                optimizer.zero_grad(set_to_none=True)
                loss.backward()
                clip_grad_norm_(model.parameters(), max_grad_norm)
                optimizer.step()
            if scheduler is not None:
                try:
                    scheduler.step()
                except Exception:
                    pass
        total_loss += loss.item()
        steps += 1
        iterator.set_postfix(loss=total_loss / max(1, steps))
    return total_loss / max(1, steps)

## Checkpoints management

In [6]:
def load_checkpoint(
    model: Seq2SeqModelWithFlashAttn,
    path: Path,
    device: torch.device,
) -> None:
    state = torch.load(path, map_location=device)
    model.load_state_dict(state["model_state_dict"])


def save_checkpoint(
    model: Seq2SeqModelWithFlashAttn,
    optimizer: torch.optim.Optimizer,
    scheduler: Optional[object],
    path: Path,
    epoch: int,
) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    state = {
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
    }
    if scheduler is not None and hasattr(scheduler, "state_dict"):
        state["scheduler_state_dict"] = scheduler.state_dict()
    torch.save(state, path)


In [7]:
# ROUGE evaluation and generation helpers
from typing import Any
from contextlib import nullcontext

try:
    from rouge_score import rouge_scorer, scoring
except Exception:
    rouge_scorer = None
    scoring = None


def _slice_sequences(flat: torch.Tensor, lengths: torch.Tensor) -> List[torch.Tensor]:
    starts = torch.cumsum(lengths, dim=0) - lengths
    seqs = []
    for i, l in enumerate(lengths.tolist()):
        seqs.append(flat[starts[i]: starts[i] + l])
    return seqs


def _decode_without_special(ids: List[int], tokenizer: PreTrainedTokenizerBase) -> str:
    bos = getattr(tokenizer, 'bos_token_id', None)
    eos = getattr(tokenizer, 'eos_token_id', None)
    pad = getattr(tokenizer, 'pad_token_id', None)
    filtered = [t for t in ids if (pad is None or t != pad) and (bos is None or t != bos) and (eos is None or t != eos)]
    return tokenizer.decode(filtered, skip_special_tokens=True)


def safe_generate(model, input_ids: torch.Tensor, src_seq_len: torch.Tensor, generation_limit: int, gen_params: Dict[str, Any]):
    # Try passing advanced params; fall back to minimal signature if not supported
    try:
        return model.generate(
            input_ids=input_ids,
            src_seq_len=src_seq_len,
            generation_limit=generation_limit,
            sampling=False,
            beam_size=gen_params.get('beam_size', 4),
            length_penalty=gen_params.get('length_penalty', 1.0),
            no_repeat_ngram_size=gen_params.get('no_repeat_ngram_size', 0),
            repetition_penalty=gen_params.get('repetition_penalty', 1.0),
            coverage_penalty=gen_params.get('coverage_penalty', 0.0),
            min_length=gen_params.get('min_length', 0),
        )
    except TypeError:
        # Fallback to original sampling interface
        return model.generate(
            input_ids=input_ids,
            src_seq_len=src_seq_len,
            generation_limit=generation_limit,
            sampling=False,
            top_k=gen_params.get('top_k', 50),
            top_p=gen_params.get('top_p', 0.9),
        )


def compute_rouge_scores(
    model: Seq2SeqModelWithFlashAttn,
    dataloader: DataLoader,
    tokenizer: PreTrainedTokenizerBase,
    device: torch.device,
    gen_params: Dict[str, Any],
    max_batches: int = 50,
) -> Dict[str, float]:
    if rouge_scorer is None:
        print('[WARN] rouge-score not available; skipping ROUGE computation.')
        return {"rouge1": float('nan'), "rouge2": float('nan'), "rougeL": float('nan')}
    scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
    aggregator = scoring.BootstrapAggregator()

    model.eval()
    batches = 0
    with torch.no_grad():
        for sample in tqdm(dataloader, desc='val-generate', leave=False):
            input_ids = sample["src"].to(device)
            src_lens = sample["src_len"].to(device=device, dtype=torch.int32)
            summaries = safe_generate(
                model,
                input_ids=input_ids,
                src_seq_len=src_lens,
                generation_limit=gen_params.get('max_length', MAX_GENERATION_LEN),
                gen_params=gen_params,
            )
            # Reconstruct targets for reference texts (supports padded or flat)
            refs: List[str]
            if 'tgt' in sample and 'tgt_len' in sample:
                tgt = sample['tgt']
                tgt_len = sample['tgt_len']
                if isinstance(tgt, torch.Tensor):
                    if tgt.dim() == 2:
                        refs = []
                        for i in range(tgt.size(0)):
                            L = int(tgt_len[i].item())
                            seq = tgt[i, :L].cpu().tolist()
                            refs.append(_decode_without_special(seq, tokenizer))
                    else:
                        tgt_flat = tgt.cpu()
                        tgt_len_cpu = tgt_len.cpu()
                        tgt_seqs = _slice_sequences(tgt_flat, tgt_len_cpu)
                        refs = [_decode_without_special(seq.tolist(), tokenizer) for seq in tgt_seqs]
                else:
                    refs = [""] * len(summaries)
            else:
                refs = [""] * len(summaries)

            for pred, ref in zip(summaries, refs):
                aggregator.add_scores(scorer.score(ref, pred))

            batches += 1
            if batches >= max_batches:
                break
    result = aggregator.aggregate()
    return {
        "rouge1": result['rouge1'].mid.fmeasure,
        "rouge2": result['rouge2'].mid.fmeasure,
        "rougeL": result['rougeL'].mid.fmeasure,
    }

## Training

In [8]:
### Hyperparameters and arguments ###
lr_decoder = 1e-4
lr_encoder = 1e-5
weight_decay = 0.001
warmup_steps = 2000
epochs = TRAIN_EPOCHS
max_grad_norm = 1.0
batch_size = TRAIN_BATCH_SIZE
num_workers = NUM_WORKERS
label_smoothing = 0.1
use_amp = True

# Auxiliary decoder pretraining (DAE)
dae_ratio = 0.15
dae_loss_weight = 0.5

# Generation/eval params
gen_params = {
    'beam_size': 4,
    'length_penalty': 1.1,
    'no_repeat_ngram_size': 3,
    'repetition_penalty': 1.1,
    'coverage_penalty': 0.0,
    'min_length': 8,
    'max_length': MAX_GENERATION_LEN,
}

# Early stopping
early_stop_metric = 'rougeL'
patience = 500
#####################################
set_seed(GLOBAL_SEED)
# Enable TF32 where available for speed
try:
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
except Exception:
    pass

if torch.cuda.is_available():
    device = torch.device("cuda:0")
else:
    raise RuntimeError("CUDA is required to run this code.")

# Check if flash attention is available
try:
    import flash_attn  # noqa: F401
except ImportError:
    raise ImportError("flash_attn is required to run this code.")

# Weights & Biases init (graceful fallback if permission/API issues)
import os, wandb, json
wandb_run = None
_wandb_err = None
try:
    wandb_run = wandb.init(project=os.environ.get('WANDB_PROJECT','lab6-summarization'),
                           entity=os.environ.get('WANDB_ENTITY'),
                           config={
                               "lr_decoder": lr_decoder,
                               "lr_encoder": lr_encoder,
                               "weight_decay": weight_decay,
                               "warmup_steps": warmup_steps,
                               "epochs": epochs,
                               "max_grad_norm": max_grad_norm,
                               "batch_size": batch_size,
                               "num_workers": num_workers,
                               "model": "ModernBERT-base",
                               "label_smoothing": label_smoothing,
                               "use_amp": use_amp,
                               "gen_params": gen_params,
                               "dae_ratio": dae_ratio,
                               "dae_loss_weight": dae_loss_weight,
                           })
except Exception as e:
    _wandb_err = e
    print('[WARN] wandb.init failed, continue without remote logging:', e)
    if os.environ.get('WANDB_API_KEY') and not os.environ.get('WANDB_MODE'):
        os.environ['WANDB_MODE'] = 'offline'
        print('[INFO] Set WANDB_MODE=offline; run will save locally.')

model = Seq2SeqModelWithFlashAttn(
    transformer_model_path="answerdotai/ModernBERT-base",
    freeze_encoder=True,
).to(device)
# Enable gradient checkpointing if supported
if hasattr(model, 'enable_gradient_checkpointing'):
    try:
        model.enable_gradient_checkpointing()
        print('[INFO] Enabled gradient checkpointing')
    except Exception as _:
        pass

if wandb_run is not None:
    wandb.watch(model, log="gradients", log_freq=100)
print(next(model.parameters()).device)
tokenizer = model.tokenizer
checkpoint_path = CHECKPOINT_PATH
best_checkpoint_path = BEST_CHECKPOINT_PATH
print('[INFO] wandb status:', 'active' if wandb_run else f'inactive ({_wandb_err})')

# Build datasets and (optionally) weighted sampler for multi-dataset training
train_set, train_sizes = build_dataset(
    ["dataset/tifu/tifu_train.jsonl", "dataset/samsun/train.csv"],
    tokenizer=model.tokenizer,
)
# Build per-sample weights to balance datasets roughly equally
train_weights = None
if isinstance(train_set, ConcatDataset) and train_sizes is not None and len(train_sizes) > 1:
    total = sum(train_sizes)
    # Equalize contribution from each dataset
    per_ds_weight = [0.5 / s if s > 0 else 0.0 for s in train_sizes]
    train_weights = []
    for w, s in zip(per_ds_weight, train_sizes):
        train_weights.extend([w] * s)

train_loader = build_dataloader(
    train_set,
    batch_size=batch_size,
    shuffle=(train_weights is None),
    num_workers=num_workers,
    sample_weights=train_weights,
)

val_set, _ = build_dataset(
    ["dataset/tifu/tifu_val.jsonl", "dataset/samsun/validation.csv"],
    tokenizer=model.tokenizer,
)
valid_loader = build_dataloader(
    val_set,
    batch_size=batch_size,
    shuffle=False,
    num_workers=num_workers,
)

# Parameter groups: lower lr for encoder, higher lr for decoder/new layers
def build_param_groups(m):
    enc_params, dec_params = [], []
    for name, p in m.named_parameters():
        if not p.requires_grad:
            continue
        if name.startswith('encoder') or 'encoder.' in name:
            enc_params.append(p)
        else:
            dec_params.append(p)
    # If encoder grouping fails (no names matched), fallback to all in dec_params
    if len(enc_params) == 0:
        dec_params = [p for p in m.parameters() if p.requires_grad]
    return [
        {"params": enc_params, "lr": lr_encoder},
        {"params": dec_params, "lr": lr_decoder},
    ]

optimizer = torch.optim.AdamW(
    build_param_groups(model), weight_decay=weight_decay
)

# Warmup + Cosine scheduler
from torch.optim.lr_scheduler import LinearLR, CosineAnnealingLR, SequentialLR

if isinstance(train_loader, DataLoader):
    total_steps = max(1, epochs * len(train_loader))
else:
    total_steps = epochs * 1000

warmup_steps = min(warmup_steps, total_steps-1) if total_steps > 1 else 0
warmup = LinearLR(optimizer, start_factor=0.1, end_factor=1.0, total_iters=max(1, warmup_steps))
cosine = CosineAnnealingLR(optimizer, T_max=max(1, total_steps - warmup_steps))
scheduler = SequentialLR(optimizer, schedulers=[warmup, cosine], milestones=[max(1, warmup_steps)])

# 選擇 AMP dtype 與是否使用 GradScaler
param_dtypes = {p.dtype for p in model.parameters() if p.requires_grad}
if torch.bfloat16 in param_dtypes and torch.float16 not in param_dtypes:
    amp_dtype = torch.bfloat16
    scaler = None  # bfloat16 不需要 GradScaler
    print('[INFO] Using bfloat16 autocast without GradScaler.')
else:
    amp_dtype = torch.float16
    scaler = torch.amp.GradScaler('cuda') if (use_amp and torch.cuda.is_available()) else None
    if scaler is not None:
        print('[INFO] Using float16 autocast with GradScaler.')

# Progressive unfreezing: unfreeze encoder after N epochs
unfreeze_at = 2

def unfreeze_encoder(m):
    if hasattr(m, 'encoder'):
        for p in m.encoder.parameters():
            p.requires_grad = True
        print('[INFO] Encoder unfrozen')
    else:
        # Best-effort: unfreeze any module name containing 'encoder'
        for name, mod in m.named_modules():
            if 'encoder' in name:
                for p in mod.parameters(recurse=False):
                    p.requires_grad = True
        print('[INFO] Best-effort encoder unfreeze applied')

best_metric = -float('inf')
no_improve = 0

for epoch in range(1, epochs + 1):
    if epoch == unfreeze_at:
        unfreeze_encoder(model)
        # Rebuild optimizer with new param groups (encoder now trainable)
        optimizer = torch.optim.AdamW(build_param_groups(model), weight_decay=weight_decay)
        # Rebuild scheduler to continue smoothly (optional simple reset)
        warmup = LinearLR(optimizer, start_factor=1.0, end_factor=1.0, total_iters=1)
        cosine = CosineAnnealingLR(optimizer, T_max=max(1, total_steps - warmup_steps))
        scheduler = SequentialLR(optimizer, [warmup, cosine], milestones=[1])

    train_loss = run_epoch(
        train_loader,
        model,
        device,
        optimizer,
        scheduler,
        tokenizer.pad_token_id,
        max_grad_norm,
        train=True,
        scaler=scaler,
        label_smoothing=label_smoothing,
        use_amp=use_amp,
        amp_dtype=amp_dtype,
        dae_ratio=dae_ratio,
        dae_loss_weight=dae_loss_weight,
    )

    with torch.no_grad():
        val_loss = run_epoch(
            valid_loader,
            model,
            device,
            optimizer=None,
            scheduler=None,
            pad_id=tokenizer.pad_token_id,
            max_grad_norm=max_grad_norm,
            train=False,
            scaler=None,
            label_smoothing=0.0,
            use_amp=False,
            amp_dtype=amp_dtype,
            dae_ratio=0.0,
            dae_loss_weight=0.0,
        )

    perplexity = math.exp(min(20, val_loss))
    rouge_scores = compute_rouge_scores(
        model, valid_loader, tokenizer, device, gen_params, max_batches=20
    )
    metric_value = rouge_scores.get(early_stop_metric, float('nan'))

    msg = (
        f"Epoch {epoch}/{epochs} - train loss: {train_loss:.4f} | "
        f"val loss: {val_loss:.4f} | ppl: {perplexity:.2f} | "
        f"R1: {rouge_scores['rouge1']:.4f} R2: {rouge_scores['rouge2']:.4f} RL: {rouge_scores['rougeL']:.4f}"
    )
    print(msg)

    try:
        wandb.log({
            "epoch": epoch,
            "train_loss": train_loss,
            "val_loss": val_loss,
            "val_perplexity": perplexity,
            "rouge1": rouge_scores['rouge1'],
            "rouge2": rouge_scores['rouge2'],
            "rougeL": rouge_scores['rougeL'],
            "lr_group_0": optimizer.param_groups[0]['lr'],
            "lr_group_1": optimizer.param_groups[1]['lr'] if len(optimizer.param_groups) > 1 else optimizer.param_groups[0]['lr'],
        })
    except Exception:
        pass

    if checkpoint_path is not None:
        save_checkpoint(
            model=model,
            optimizer=optimizer,
            scheduler=scheduler,
            path=checkpoint_path,
            epoch=epoch,
        )

    improved = metric_value > best_metric
    if improved:
        best_metric = metric_value
        no_improve = 0
        if best_checkpoint_path is not None:
            save_checkpoint(
                model=model,
                optimizer=optimizer,
                scheduler=scheduler,
                path=best_checkpoint_path,
                epoch=epoch,
            )
    else:
        no_improve += 1
        if no_improve >= patience:
            print(f"[EARLY STOP] No improvement in {early_stop_metric} for {patience} epochs. Stop.")
            break

wandb: Currently logged in as: aaronwu901225main (NYCU_Deeplearning) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


wandb: setting up run x1hri14e


wandb: Tracking run with wandb version 0.23.0


wandb: Run data is saved locally in /home/at0842/aaronwu901225master.ai13/sundries/lab6/wandb/run-20251126_234558-x1hri14e
wandb: Run `wandb offline` to turn off syncing.


wandb: Syncing run apricot-sponge-18


...

Epoch 199/200 - train loss: 5.3480 | val loss: 4.2543 | ppl: 70.41 | R1: 0.2222 R2: 0.0424 RL: 0.1743


train:   0%|          | 0/346 [00:00<?, ?it/s]

eval:   0%|          | 0/40 [00:00<?, ?it/s]

val-generate:   0%|          | 0/40 [00:00<?, ?it/s]

Epoch 200/200 - train loss: 5.3551 | val loss: 4.2543 | ppl: 70.41 | R1: 0.2225 R2: 0.0426 RL: 0.1748


## Predict Result

Predict the labesl based on testing set. Upload to [Kaggle](https://www.kaggle.com/t/efb569a4c0774de681e9f8426cfac364).

**How to upload**

1. To kaggle. Click "Submit Predictions"
2. Upload the result.csv
3. System will automaticlaly calculate the accuracy of 50% dataset and publish this result to leaderboard.

In [9]:
load_checkpoint(model, PREDICT_CHECKPOINT, device)
model.eval()

test_set, _ = build_dataset(
    [TIFU_TEST_PATH, SAMSUN_TEST_PATH],
    tokenizer=model.tokenizer,
    require_target=False,
)

test_loader = build_dataloader(
    test_set,
    batch_size=batch_size,
    shuffle=False,
    num_workers=num_workers,
)

# Use beam search with repetition control for more stable predictions
infer_gen_params = {
    'beam_size': 4,
    'length_penalty': 1.1,
    'no_repeat_ngram_size': 3,
    'repetition_penalty': 1.1,
    'coverage_penalty': 0.0,
    'min_length': 8,
    'max_length': MAX_GENERATION_LEN,
}

predictions: List[Tuple[str, str]] = []
with torch.no_grad():
    for sample in tqdm(test_loader, desc="predict", leave=False):
        input_ids = sample["src"].to(device)
        src_lens = sample["src_len"].to(device=device, dtype=torch.int32)
        ids = sample["id"]  # list of ids
        summaries = safe_generate(
            model,
            input_ids=input_ids,
            src_seq_len=src_lens,
            generation_limit=infer_gen_params.get('max_length', MAX_GENERATION_LEN),
            gen_params=infer_gen_params,
        )
        predictions.extend(zip(ids, summaries))

output_path = PREDICTION_OUTPUT
write_predictions_csv(output_path, predictions)
print(f"Wrote {len(predictions)} predictions to {output_path}")


Built dataset with 9248 samples.


predict:   0%|          | 0/73 [00:00<?, ?it/s]

Wrote 9248 predictions to result.csv
